# Grokking Research Lab

## Setup

In [ ]:
import os, sys, json, math, time, random, platform, subprocess, hashlib
from pathlib import Path
from dataclasses import dataclass, asdict, replace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F

RUN_PILOT = True
RUN_SWEEP = False
FINAL_PROTOCOL_FROZEN = False
SMOKE_STEPS = 100

if Path("/kaggle/working").exists():
    ROOT = Path("/kaggle/working/grokking_lab")
elif Path("/content").exists():
    ROOT = Path("/content/grokking_lab")
else:
    ROOT = Path("runs")
ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(2)


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


environment = {
    "python": sys.version,
    "torch": str(torch.__version__),
    "numpy": np.__version__,
    "device": str(DEVICE),
    "platform": platform.platform(),
    "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
}
(ROOT / "environment.json").write_text(json.dumps(environment, indent=2))
(ROOT / "pip_freeze.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
)
print(json.dumps(environment, indent=2))

## Dataset and configuration

In [ ]:
@dataclass(frozen=True)
class Config:
    p: int = 113
    d_model: int = 128
    n_heads: int = 4
    d_mlp: int = 512
    train_fraction: float = 0.30
    split_seed: int = 2026
    seed: int = 0
    lr: float = 1e-3
    weight_decay: float = 1.0
    steps: int = 30000
    log_every: int = 100
    checkpoint_every: int = 1000


cfg = Config()


def make_data(c):
    a, b = torch.meshgrid(torch.arange(c.p), torch.arange(c.p), indexing="ij")
    x = torch.stack([a.flatten(), b.flatten(), torch.full((c.p * c.p,), c.p)], dim=1)
    y = (x[:, 0] + x[:, 1]) % c.p
    g = torch.Generator().manual_seed(c.split_seed)
    order = torch.randperm(len(y), generator=g)
    n_holdout = int(0.20 * len(y))
    val = order[:n_holdout]
    test = order[n_holdout : 2 * n_holdout]
    pool = order[2 * n_holdout :]
    n_train = int(c.train_fraction * len(y))
    assert 0 < n_train <= len(pool)
    train = pool[:n_train]
    assert not set(train.tolist()) & set(val.tolist())
    assert not set(train.tolist()) & set(test.tolist())
    assert not set(val.tolist()) & set(test.tolist())
    return x, y, {"train": train, "val": val, "test": test, "pool": pool}


x, y, split = make_data(cfg)
assert y[110 * cfg.p + 7].item() == 4
assert len(torch.unique(x[:, :2], dim=0)) == cfg.p**2
print("Input shape:", x.shape, "Target shape:", y.shape)
print({k: len(v) for k, v in split.items()})
print("Example:", x[110 * cfg.p + 7].tolist(), "target:", y[110 * cfg.p + 7].item())

## Transformer

In [ ]:
class TinyTransformer(nn.Module):
    def __init__(self, c):
        super().__init__()
        assert c.d_model % c.n_heads == 0
        self.c = c
        self.token = nn.Embedding(c.p + 1, c.d_model)
        self.pos = nn.Parameter(torch.empty(3, c.d_model))
        self.qkv = nn.Linear(c.d_model, 3 * c.d_model, bias=False)
        self.out = nn.Linear(c.d_model, c.d_model, bias=False)
        self.fc1 = nn.Linear(c.d_model, c.d_mlp)
        self.fc2 = nn.Linear(c.d_mlp, c.d_model)
        self.unembed = nn.Linear(c.d_model, c.p, bias=False)
        self.register_buffer(
            "causal", torch.triu(torch.ones(3, 3, dtype=torch.bool), diagonal=1)
        )
        self.apply(self.initialize)
        nn.init.normal_(self.pos, std=0.02)

    @staticmethod
    def initialize(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward(self, tokens, ablate_heads=(), return_cache=False):
        B, T = tokens.shape
        H, D = self.c.n_heads, self.c.d_model // self.c.n_heads
        embedded = self.token(tokens) + self.pos[:T]
        qkv = self.qkv(embedded).reshape(B, T, 3, H, D).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        scores = (q @ k.transpose(-1, -2)) / math.sqrt(D)
        scores = scores.masked_fill(self.causal[:T, :T], float("-inf"))
        attention = scores.softmax(dim=-1)
        heads = attention @ v
        if ablate_heads:
            mask = torch.ones(H, device=tokens.device)
            mask[list(ablate_heads)] = 0
            heads = heads * mask[None, :, None, None]
        merged = heads.transpose(1, 2).reshape(B, T, H * D)
        resid = embedded + self.out(merged)
        mlp_hidden = F.relu(self.fc1(resid))
        final = resid + self.fc2(mlp_hidden)
        logits = self.unembed(final[:, -1])
        if return_cache:
            return logits, {
                "attention": attention.detach(),
                "mlp_last": mlp_hidden[:, -1].detach(),
                "final_last": final[:, -1].detach(),
            }
        return logits


seed_all(cfg.seed)
model = TinyTransformer(cfg).to(DEVICE)
print("Trainable parameters:", sum(v.numel() for v in model.parameters()))
with torch.no_grad():
    probe, cache = model(x[:8].to(DEVICE), return_cache=True)
assert probe.shape == (8, cfg.p)
assert torch.isfinite(probe).all()
assert torch.allclose(
    cache["attention"].sum(-1), torch.ones_like(cache["attention"].sum(-1)), atol=1e-6
)
assert cache["attention"][:, :, 0, 1:].abs().max().item() == 0
assert torch.equal(y, (x[:, 0] + x[:, 1]) % cfg.p)
print("Dataset, shapes, finite outputs, and causal mask checks passed.")

## Training and checkpoints

In [ ]:
@torch.no_grad()
def evaluate(model, x_cpu, y_cpu, ids, ablate_heads=(), batch_size=1024):
    model.eval()
    loss_sum = 0.0
    correct = 0
    for chunk in ids.split(batch_size):
        xx, yy = x_cpu[chunk].to(DEVICE), y_cpu[chunk].to(DEVICE)
        logits = model(xx, ablate_heads=ablate_heads)
        loss_sum += F.cross_entropy(logits, yy, reduction="sum").item()
        correct += (logits.argmax(-1) == yy).sum().item()
    return {"loss": loss_sum / len(ids), "accuracy": correct / len(ids)}


def run(c, name, resume=False):
    folder = ROOT / name
    folder.mkdir(exist_ok=True)
    if any(folder.iterdir()) and not resume:
        raise FileExistsError(
            f"{folder} already has a run. Choose another name or resume=True."
        )
    seed_all(c.seed)
    xx, yy, ss = make_data(c)
    m = TinyTransformer(c).to(DEVICE)
    optimizer = torch.optim.AdamW(
        m.parameters(), lr=c.lr, betas=(0.9, 0.98), weight_decay=c.weight_decay
    )
    history, start, previous_seconds = [], 0, 0.0
    if resume:
        # Load only checkpoints you created and trust.
        ck = torch.load(folder / "latest.pt", map_location=DEVICE, weights_only=False)
        assert ck["config"] == asdict(c), "Resume with the same configuration."
        m.load_state_dict(ck["model"])
        optimizer.load_state_dict(ck["optimizer"])
        history, start = ck["history"], ck["step"]
        previous_seconds = ck["elapsed_seconds"]
    (folder / "config.json").write_text(json.dumps(asdict(c), indent=2))
    np.savez(folder / "split_indices.npz", **{k: v.numpy() for k, v in ss.items()})
    train_x, train_y = xx[ss["train"]].to(DEVICE), yy[ss["train"]].to(DEVICE)
    t0 = time.perf_counter()

    def elapsed():
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        return previous_seconds + time.perf_counter() - t0

    def log(step):
        tr = evaluate(m, xx, yy, ss["train"])
        va = evaluate(m, xx, yy, ss["val"])
        norm = sum(w.detach().square().sum().item() for w in m.parameters()) ** 0.5
        row = {
            "step": step,
            "train_loss": tr["loss"],
            "val_loss": va["loss"],
            "train_acc": tr["accuracy"],
            "val_acc": va["accuracy"],
            "weight_norm": norm,
            "elapsed_seconds": elapsed(),
        }
        history.append(row)
        pd.DataFrame(history).to_csv(folder / "history.csv", index=False)
        print({k: round(v, 4) if isinstance(v, float) else v for k, v in row.items()})

    def save(step, snapshot=False):
        checkpoint = {
            "config": asdict(c),
            "model": m.state_dict(),
            "optimizer": optimizer.state_dict(),
            "step": step,
            "history": history,
            "elapsed_seconds": elapsed(),
        }
        tmp = folder / "temporary.pt"
        torch.save(checkpoint, tmp)
        tmp.replace(folder / "latest.pt")
        if snapshot:
            torch.save(
                {"config": asdict(c), "model": m.state_dict(), "step": step},
                folder / f"model_step_{step:06d}.pt",
            )

    if start == 0:
        log(0)
        save(0, snapshot=True)
    for step in range(start + 1, c.steps + 1):
        m.train()
        optimizer.zero_grad(set_to_none=True)
        loss = F.cross_entropy(m(train_x), train_y)
        if not torch.isfinite(loss):
            raise RuntimeError(
                "Nonfinite loss: inspect learning rate and initialization."
            )
        loss.backward()
        optimizer.step()
        if step % c.log_every == 0 or step == c.steps:
            log(step)
        if step % c.checkpoint_every == 0 or step == c.steps:
            save(step, snapshot=True)
    return m, pd.DataFrame(history), folder

## Smoke run and pilot

In [ ]:
smoke_model, smoke_history, smoke_dir = run(
    replace(
        cfg,
        steps=SMOKE_STEPS,
        log_every=max(1, SMOKE_STEPS // 2),
        checkpoint_every=SMOKE_STEPS,
    ),
    "smoke_seed0",
    resume=(ROOT / "smoke_seed0" / "latest.pt").exists(),
)
print("Files:", [p.name for p in smoke_dir.iterdir()])
seconds = smoke_history.iloc[-1]["elapsed_seconds"]
print(
    "Rough pilot hours (includes smoke overhead):",
    seconds / SMOKE_STEPS * cfg.steps / 3600,
)

In [ ]:
if RUN_PILOT:
    if DEVICE.type != "cuda":
        raise RuntimeError(
            "Enable a GPU in Colab Runtime settings or Kaggle Accelerator settings. For CPU smoke only, set RUN_PILOT = False in Setup."
        )
    pilot_model, pilot_history, pilot_dir = run(
        cfg, "pilot_seed0", resume=(ROOT / "pilot_seed0" / "latest.pt").exists()
    )

## Learning curves

In [ ]:
def plot_history(h, folder):
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].plot(h.step, h.train_loss, label="train")
    ax[0].plot(h.step, h.val_loss, label="validation")
    ax[0].set_yscale("log")
    ax[0].set_ylabel("Cross-entropy")
    ax[1].plot(h.step, h.train_acc, label="train")
    ax[1].plot(h.step, h.val_acc, label="validation")
    ax[1].set_ylabel("Accuracy")
    ax[1].set_ylim(-0.02, 1.02)
    ax[2].plot(h.step, h.weight_norm)
    ax[2].set_ylabel("Parameter L2 norm")
    for a in ax:
        a.set_xlabel("Optimizer updates")
        a.grid(alpha=0.2)
    ax[0].legend()
    ax[1].legend()
    fig.tight_layout()
    fig.savefig(folder / "learning_curves.png", dpi=160)
    plt.show()


def first_sustained(h, column, threshold=0.99, window=3):
    values = h[column].to_numpy() >= threshold
    for i in range(len(values) - window + 1):
        if values[i : i + window].all():
            return int(h.iloc[i].step)
    return None


def summarize(h):
    t = first_sustained(h, "train_acc")
    v = first_sustained(h, "val_acc")
    return {
        "train_crossing": t,
        "validation_crossing": v,
        "delay_updates": v - t if t is not None and v is not None else None,
        "last_validation_accuracy": float(h.iloc[-1].val_acc),
        "elapsed_seconds": float(h.iloc[-1].elapsed_seconds),
    }


plot_history(smoke_history, smoke_dir)
print(summarize(smoke_history))
# After the pilot finishes:
# plot_history(pilot_history,pilot_dir)
# print(summarize(pilot_history))

## Multi-seed experiments

In [ ]:
conditions = sorted(
    set([(0.30, w) for w in (0.0, 0.1, 1.0)] + [(f, 1.0) for f in (0.10, 0.30, 0.50)])
)
rows = []
if RUN_SWEEP:
    for fraction, decay in conditions:
        for seed in (0, 1, 2):
            c = replace(cfg, train_fraction=fraction, weight_decay=decay, seed=seed)
            name = f"fraction{fraction:.2f}_wd{decay:.2f}_seed{seed}"
            folder = ROOT / name
            if (folder / "latest.pt").exists():
                checkpoint = torch.load(
                    folder / "latest.pt", map_location="cpu", weights_only=False
                )
                assert checkpoint["config"] == asdict(c)
                if checkpoint["step"] == c.steps:
                    h = pd.read_csv(folder / "history.csv")
                else:
                    m, h, folder = run(c, name, resume=True)
            else:
                m, h, folder = run(c, name)
            rows.append(
                {
                    "run": name,
                    "fraction": fraction,
                    "weight_decay": decay,
                    "seed": seed,
                    "example_presentations": int(fraction * c.p * c.p) * c.steps,
                    **summarize(h),
                }
            )
            pd.DataFrame(rows).to_csv(ROOT / "sweep_summary.csv", index=False)
    display(pd.DataFrame(rows))

## Checkpoint analysis

In [ ]:
ANALYSIS_DIR = pilot_dir if RUN_PILOT else smoke_dir
ANALYSIS_STEP = cfg.steps if RUN_PILOT else SMOKE_STEPS


def load_snapshot(folder, step):
    ck = torch.load(
        folder / f"model_step_{step:06d}.pt", map_location="cpu", weights_only=True
    )
    c = Config(**ck["config"])
    m = TinyTransformer(c).to(DEVICE)
    m.load_state_dict(ck["model"])
    m.eval()
    return m, c


analysis_model, analysis_cfg = load_snapshot(ANALYSIS_DIR, ANALYSIS_STEP)
ax_data, ay_data, analysis_split = make_data(analysis_cfg)
with torch.no_grad():
    example = torch.tensor([[110, 7, analysis_cfg.p]], device=DEVICE)
    logits, cache = analysis_model(example, return_cache=True)
print("Prediction:", logits.argmax(-1).item(), "correct answer:", 4)
fig, axes = plt.subplots(1, analysis_cfg.n_heads, figsize=(12, 3))
for head, axis in enumerate(axes):
    axis.imshow(cache["attention"][0, head].cpu(), vmin=0, vmax=1)
    axis.set_title(f"Head {head}")
    axis.set_xticks(range(3), ["110", "7", "="])
    axis.set_yticks(range(3), ["110", "7", "="])
    axis.set_xlabel("Key position")
axes[0].set_ylabel("Query position")
fig.tight_layout()
fig.savefig(ANALYSIS_DIR / "attention_example.png", dpi=160)
plt.show()

## Embedding spectrum

In [ ]:
@torch.no_grad()
def embedding_spectrum(m):
    embedding = m.token.weight[: m.c.p].detach().float().cpu()
    fourier = torch.fft.rfft(embedding, dim=0, norm="ortho")
    return fourier.abs().square().sum(dim=1)


energy = embedding_spectrum(analysis_model)
plt.figure(figsize=(10, 3))
plt.bar(range(len(energy)), energy.numpy())
plt.xlabel("Frequency k")
plt.ylabel("Embedding Fourier energy")
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / "embedding_spectrum.png", dpi=160)
plt.show()

## Frequency probe

In [ ]:
@torch.no_grad()
def collect_mlp(m, xx, batch_size=512):
    pieces = []
    for batch in xx.split(batch_size):
        _, c = m(batch.to(DEVICE), return_cache=True)
        pieces.append(c["mlp_last"].cpu())
    return torch.cat(pieces).double()


def probe_frequency(m, xx, yy, ss, k=1, alpha=1e-2, shuffle=False):
    hidden = collect_mlp(m, xx)
    H = torch.cat([hidden, torch.ones(len(hidden), 1, dtype=torch.double)], dim=1)
    angle = 2 * math.pi * k * yy.double() / m.c.p
    targets = torch.stack([torch.cos(angle), torch.sin(angle)], dim=1)
    tr, va = ss["train"], ss["val"]
    train_targets = targets[tr]
    if shuffle:
        g = torch.Generator().manual_seed(123)
        train_targets = train_targets[torch.randperm(len(tr), generator=g)]
    A = H[tr]
    W = torch.linalg.solve(
        A.T @ A + alpha * torch.eye(A.shape[1], dtype=A.dtype), A.T @ train_targets
    )
    pred = H[va] @ W
    sse = (pred - targets[va]).square().sum()
    sst = (targets[va] - targets[va].mean(0)).square().sum()
    return float(1 - sse / sst)


print(
    "Frequency 1 validation R2:",
    probe_frequency(analysis_model, ax_data, ay_data, analysis_split),
)
print(
    "Shuffled-label control R2:",
    probe_frequency(analysis_model, ax_data, ay_data, analysis_split, shuffle=True),
)

## Head ablations

In [ ]:
base = evaluate(analysis_model, ax_data, ay_data, analysis_split["val"])
ablations = [{"intervention": "none", **base, "accuracy_drop": 0.0}]
for head in range(analysis_cfg.n_heads):
    result = evaluate(
        analysis_model, ax_data, ay_data, analysis_split["val"], ablate_heads=(head,)
    )
    ablations.append(
        {
            "intervention": f"remove_head_{head}",
            **result,
            "accuracy_drop": base["accuracy"] - result["accuracy"],
        }
    )
result = evaluate(
    analysis_model,
    ax_data,
    ay_data,
    analysis_split["val"],
    ablate_heads=tuple(range(analysis_cfg.n_heads)),
)
ablations.append(
    {
        "intervention": "remove_all_heads",
        **result,
        "accuracy_drop": base["accuracy"] - result["accuracy"],
    }
)
ablation_table = pd.DataFrame(ablations)
ablation_table.to_csv(ANALYSIS_DIR / "head_ablations_validation.csv", index=False)
display(ablation_table)

## Fourier interventions

In [ ]:
import copy


@torch.no_grad()
def project_embeddings(m, keep):
    changed = copy.deepcopy(m)
    E = m.token.weight[: m.c.p].detach()
    spectrum = torch.fft.rfft(E.float(), dim=0, norm="ortho")
    mask = torch.zeros(spectrum.shape[0], device=E.device)
    mask[list(keep)] = 1
    projected = torch.fft.irfft(spectrum * mask[:, None], n=m.c.p, dim=0, norm="ortho")
    changed.token.weight[: m.c.p].copy_(projected.to(E.dtype))
    retained = float(projected.square().sum() / E.square().sum())
    return changed, retained


K = min(5, len(energy) - 1)
top = (torch.topk(energy[1:], K).indices + 1).tolist()
all_nonzero = list(range(1, len(energy)))
sets = {
    "all": list(range(len(energy))),
    "keep_top": [0] + top,
    "remove_top": [0] + [k for k in all_nonzero if k not in top],
}
g = torch.Generator().manual_seed(999)
for trial in range(5):
    random_k = (torch.randperm(len(energy) - 1, generator=g)[:K] + 1).tolist()
    sets[f"keep_random_{trial}"] = [0] + random_k
frequency_rows = []
for label, keep in sets.items():
    changed, retained = project_embeddings(analysis_model, keep)
    metrics = evaluate(changed, ax_data, ay_data, analysis_split["val"])
    frequency_rows.append(
        {
            "condition": label,
            "kept_frequencies": str(keep),
            "retained_embedding_energy": retained,
            **metrics,
        }
    )
    if label == "all":
        assert abs(metrics["loss"] - base["loss"]) < 1e-4, (
            "FFT round-trip control failed."
        )
    del changed
frequency_table = pd.DataFrame(frequency_rows)
frequency_table.to_csv(
    ANALYSIS_DIR / "frequency_interventions_validation.csv", index=False
)
display(frequency_table)

## Final test

In [ ]:
if FINAL_PROTOCOL_FROZEN:
    assert ANALYSIS_STEP == analysis_cfg.steps, (
        "Use the final fixed-horizon checkpoint."
    )
    result = evaluate(analysis_model, ax_data, ay_data, analysis_split["test"])
    report = {
        "config": asdict(analysis_cfg),
        "checkpoint_step": ANALYSIS_STEP,
        "test": result,
        "status": "measured by executing this notebook",
    }
    (ANALYSIS_DIR / "final_test.json").write_text(json.dumps(report, indent=2))
    print(report)

## Download results

In [ ]:
import shutil

archive = shutil.make_archive(
    str(ROOT.parent / "grokking_lab_outputs"), "zip", ROOT.parent, ROOT.name
)
print("Results and checkpoints:", archive)
try:
    from google.colab import files
except ImportError:
    try:
        from IPython.display import FileLink, display
    except ImportError:
        pass
    else:
        display(FileLink(archive))
else:
    files.download(archive)